# Experiment Web Application Dashboard

# Introduction

This is the capstone of Project 7. Everything you've built — MongoDB
queries (L1), ETL pipelines (L2), power analysis and the chi-square test
(L3) — now gets assembled into a single, cohesive **web application**.
The dashboard lets a user explore applicant demographics, configure an
A/B experiment, launch it, and read the statistical verdict, all without
writing a line of code.

The real lesson here is **architecture**: how to organise a non-trivial
application so it stays understandable as it grows. You'll build it in
three clean layers and watch them snap together.

> 🎯 **By the end of this notebook you will be able to:**
>
> - Explain the three-tier architecture pattern (database, business
>   logic, presentation) and why it helps organize interactive
>   applications.
> - Build a `MongoRepository` class that encapsulates all MongoDB queries
>   needed by the dashboard.
> - Create reusable chart-building and statistics-computing classes
>   (`GraphBuilder`, `StatsBuilder`) that rely on the repository for data.
> - Construct a Dash application layout with drop-downs, sliders, buttons,
>   and display areas.
> - Wire Dash callbacks so that user interactions trigger data retrieval,
>   computation, and visualization updates.
> - Launch and interact with the finished dashboard inline inside a
>   Jupyter notebook.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183285103", h="3298dbabb7", width=700, height=450) 

# 1. Conceptual Foundation

## Project context: what we're trying to build

Across Project 7 you explored applicant data in MongoDB (Lesson 1), built
ETL pipelines (Lesson 2), and ran hypothesis tests (Lesson 3). This final
lesson fuses those pieces into an **Experiment Monitoring Dashboard** — a
single interactive app with three sections:

| Section | What the user does | Powered by |
|---|---|---|
| **Applicant Demographics** | Pick a chart (nationality map, age histogram, education bar) from a drop-down | `GraphBuilder` |
| **Experiment** | Set effect size and duration with sliders; see required sample size and the probability of collecting it | `StatsBuilder` |
| **Results** | Click a button to run the experiment; see the contingency bar chart and chi-square verdict | `StatsBuilder` + `GraphBuilder` |

## Three-tier architecture

Professional web applications almost always split their code into three
layers. The discipline isn't bureaucracy — it's what keeps a growing app
from collapsing into a tangle where every change risks breaking
everything else.

| Layer | Responsibility | In this lesson |
|---------|-------------|---------------------------------------------------|
| **Database** | Data access and storage | `MongoRepository` class — queries MongoDB and returns pandas objects |
| **Business logic** | Computation and transformation | `GraphBuilder` (builds Plotly figures) and `StatsBuilder` (sample-size calculations, CDF, experiment runner, chi-square test) |
| **Presentation** | User interface and interaction | Dash layout (HTML components) and callbacks |

🧠 **The key rule is one-directional flow.** Information moves top-down: a
callback in the **presentation** layer creates a **business-logic**
builder, which internally asks the **database** layer for data. The
presentation layer never touches `pymongo` directly, and the database
layer knows nothing about Dash. Each layer can be tested — and
swapped — in isolation.

➡️ In a production project each layer lives in its own module
(`database.py`, `business.py`, `display.py`) so it can be imported and
reused across many notebooks. Here you'll define all three *inside the
notebook* so you can build and test each piece incrementally, with
immediate cell-by-cell feedback — but the boundaries between them stay
just as crisp.

## The `Experiment` helper class

In Lesson 3 you used the `Experiment` class from
`wqulibs.ab_test.experiment` to simulate the A/B test. It's a pre-built
helper — you don't modify it. It generates synthetic applicants day by
day, randomly assigns each to control (`"no email (control)"`) or
treatment (`"email (treatment)"`), and inserts the documents into MongoDB.

In this lesson the `StatsBuilder` class uses `Experiment` internally to
power the "Run Experiment" button. The methods you'll call:

- `Experiment(repo=...)` — create an instance connected to a
  `MongoRepository` (or a `MongoClient`).
- `exp.reset_experiment()` — remove any previous experiment data.
- `exp.run_experiment(days=n)` — simulate `n` days and insert the
  generated documents.

## Dash overview

Dash is a Python framework for building interactive web apps. A Dash app
has exactly two parts:

- A **layout** — a tree of components defining what the user sees
  (headings, drop-downs, sliders, buttons, charts, text).
- **Callbacks** — Python functions that run automatically when the user
  interacts with a component. Each callback declares which component
  properties it reads (`Input`) and which it writes (`Output`).

🔄 **The callback mental model:** *"When this `Input` changes, run my
function and put the return value into that `Output`."* Dash watches the
inputs, and whenever one changes it re-runs the matching callback and
patches the output into the page. You never write the event-loop
plumbing — you just declare the wiring. Here is the minimal pattern:

In [ ]:
# Conceptual sketch — do not run this cell.
# It illustrates the Dash layout + callback pattern.
# ---------------------------------------------------
# from dash import Dash, html, dcc, Input, Output
#
# app = Dash(__name__)
#
# app.layout = html.Div([
#     html.H1("Demo"),
#     dcc.Dropdown(
#         id="color-dropdown",
#         options=["Red", "Blue", "Green"],
#         value="Red",
#     ),
#     html.Div(id="color-output"),
# ])
#
# @app.callback(
#     Output("color-output", "children"),
#     Input("color-dropdown", "value"),
# )
# def show_color(selection):
#     return f"You picked: {selection}"

📌 Every interactive component needs a unique **`id`** string — that's the
name callbacks use to wire an `Input` or `Output` to a specific component.

## Key Dash components used in this lesson

| Component | Role |
|---|---|
| `html.H1` | Section headings |
| `dcc.Dropdown` | Pick one option from a list |
| `dcc.Slider` | Pick a numeric value along a range |
| `html.Button` | Trigger an action when clicked |
| `dcc.Graph` | Render a Plotly figure |
| `html.Div` | Generic container; common callback-output target |

## Debugging tips for Dash apps

Building a Dash app inside a notebook is convenient, but a few things can
trip you up:

- ⚠️ **Callback errors produce blank output.** If a callback raises an
  exception, its target `Div` or `Graph` just stays empty — no red error
  on the page. Check the cell output or terminal for the traceback.
- **Layout before callbacks.** Dash registers callbacks against component
  IDs *in the layout*. A callback that references an ID not present in the
  layout errors at registration time.
- **Restart when stuck.** If the app misbehaves, restart the kernel
  (`Kernel → Restart Kernel and Clear All Outputs`) and rerun all cells
  from the top.
- **One app per kernel.** Only one Dash server can run at a time in a
  notebook. To rebuild the app, restart the kernel first.

> 📚 **Key references for this lesson:**
>
> - [`dash.Dash`](https://dash.plotly.com/reference#dash.dash) — the main
>   application class.
> - [`dash.dcc`](https://dash.plotly.com/dash-core-components) —
>   high-level interactive components (Dropdown, Slider, Graph).
> - [`dash.html`](https://dash.plotly.com/dash-html-components) — HTML
>   wrapper components.
> - [`dash.Input` /
>   `dash.Output`](https://dash.plotly.com/basic-callbacks) — callback
>   decorators.
> - [`plotly.express`](https://plotly.com/python/plotly-express/) — quick
>   chart-building API.
> - [`pymongo.collection.Collection`](https://pymongo.readthedocs.io/en/stable/api/pymongo/collection.html)
>   — MongoDB collection interface.
> - [`statsmodels.stats.contingency_tables`](https://www.statsmodels.org/stable/contingency_tables.html)
>   — chi-square test utilities.

# Applied Exercises

## 2. Setup

Note: Import the required libraries and classes. It is highly recommended
to place all imports in a single cell at the beginning of the notebook.

**🛠️ Instruction:** Locate the IP address of the machine running MongoDB
and assign it to the variable `MONGODB_HOST`. Make sure to use a
**string** (i.e., wrap the IP in quotes).

**⚠️ Note:** The IP address is **dynamic** — it may change every time you
start the lab. Always check the current IP before proceeding.

<figure>
<img src="attachment:images/mongo_ip.png" alt="MongoDB" />
<figcaption aria-hidden="true">MongoDB</figcaption>
</figure>

**Code 7.4.2.1**:

In [ ]:
import pandas as pd
import plotly.express as px
from country_converter import CountryConverter
from dash import Dash, html, dcc, Input, Output, State
from pymongo import MongoClient

from wqulibs.database import reset

MONGODB_HOST = "192.95.222.3"

## 3. Database Layer — MongoRepository

### Problem

Every visualization and statistical computation in the dashboard needs
data from MongoDB. Rather than scattering database queries throughout the
notebook, you will create a single `MongoRepository` class that
encapsulates all data access. The rest of the code will call repository
methods and receive clean pandas objects, never touching `pymongo`
directly.

In a production project this class would live in its own module (e.g.,
`database.py`) so that other notebooks or scripts can import and reuse it.
Here you will define it in a notebook cell, but the design principle is
the same.

🧱 **This is the bottom tier.** It is the *only* layer that speaks
`pymongo`. Everything above it receives tidy DataFrames and Series, so if
the storage backend ever changed, this is the only class you'd rewrite.

### Approach

Define a `MongoRepository` class with:

-   `__init__` — connect to the local MongoDB instance, select the
    appropriate database, and store a reference to the `"ds-applicants"`
    collection as `self.collection`.
-   `get_nationality_value_counts(normalize)` — run an aggregation
    pipeline and return a DataFrame with columns `country_iso2`,
    `country_iso3`, `country_name`, and `count`.
-   `get_ages()` — return a pandas Series of applicant ages.
-   `get_ed_value_counts(normalize)` — return a Series of education-level
    counts ordered from lowest to highest degree.
-   `get_no_quiz_per_day()` — return a Series with the number of no-quiz
    applicants per day (30 entries).
-   `get_contingency_table()` — return a 2×2 DataFrame (contingency table)
    for the experiment groups.

Reuse the query logic you developed in Lessons 7.1–7.3. Refer to the
[data-dictionary.ipynb](data-dictionary.ipynb) notebook for field names
and expected values.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183285220", h="3298dbabb7", width=700, height=450) 

### Tasks

Define the `MongoRepository` class. After running this cell you should be
able to instantiate the class and confirm it connects to the correct
collection.

💡 Notice how each method maps onto a query you've already written: the
nationality `$group`, the `$dateDiff` age calculation, the `$dateTrunc`
daily counts. The class doesn't invent new logic — it **organises**
familiar queries behind named methods.

**Code Task 7.4.3.1**:

In [ ]:
class MongoRepository:
    """Provides data access methods for the ds-applicants collection.

    This class encapsulates all MongoDB queries needed by the
    dashboard. It connects to a local MongoDB instance on
    construction and exposes methods that return clean pandas
    objects for demographics, quiz statistics, and experiment
    results.

    Attributes
    ----------
    collection : pymongo.collection.Collection
        Reference to the ``ds-applicants`` collection in the
        ``wqu-abtest`` database.
    """

    def __init__(self):
        """Connect to local MongoDB and store the collection.

        Creates a ``MongoClient`` pointed at ``<MONGODB_HOST>:27017``,
        selects the ``wqu-abtest`` database, and stores the
        ``ds-applicants`` collection as ``self.collection``.
        """
        ...

    def get_nationality_value_counts(self, normalize=True):
        """Aggregate applicant counts by nationality.

        Runs a MongoDB aggregation pipeline that groups documents
        by the ``countryISO2`` field and counts each group. The
        results are enriched with ISO-3 codes and full country
        names via the ``country_converter`` library.

        Parameters
        ----------
        normalize : bool, optional
            If ``True`` (default), divide each count by the total
            so that values represent proportions. If ``False``,
            return raw counts.

        Returns
        -------
        pandas.DataFrame
            DataFrame with columns ``country_iso2``,
            ``country_iso3``, ``country_name``, and ``count``.
            Rows are sorted by ``count`` in descending order.
        """
        ...

    def get_ages(self):
        """Compute applicant ages from their birth dates.

        Uses a ``$dateDiff`` aggregation to calculate the
        difference in years between each applicant's
        ``birthday`` field and the current date (``$$NOW``).

        Returns
        -------
        pandas.Series
            A Series of integer ages, one per applicant, with
            ``NaN`` values dropped.
        """
        ...

    def get_ed_value_counts(self, normalize=False):
        """Count applicants by highest degree earned.

        Queries the ``highestDegreeEarned`` field and returns
        value counts reindexed in ascending educational order.

        Parameters
        ----------
        normalize : bool, optional
            If ``True``, return proportions instead of raw
            counts. Default is ``False``.

        Returns
        -------
        pandas.Series
            Counts (or proportions) indexed by degree name,
            ordered from ``"High School or Baccalaureate"`` to
            ``"Doctorate (e.g. PhD)"``.
        """
        ...

    def get_no_quiz_per_day(self):
        """Count daily new accounts that never completed the quiz.

        Filters for documents where ``admissionsQuiz`` is
        ``"incomplete"``, groups them by day using ``$dateTrunc``
        on ``createdAt``, and returns the first 30 days sorted
        chronologically.

        Returns
        -------
        pandas.Series
            A 30-element Series indexed by date with daily
            counts as values.
        """
        ...

    def get_contingency_table(self):
        """Build a 2x2 contingency table from experiment data.

        Queries documents where ``inExperiment`` is ``True`` and
        ``group`` is either ``"no email (control)"`` or
        ``"email (treatment)"``. Groups by ``group`` and
        ``admissionsQuiz`` status, then pivots into a 2x2
        DataFrame suitable for a chi-square test.

        Returns
        -------
        pandas.DataFrame
            A pivot table with experiment groups as the index
            and quiz completion status as the columns, with
            counts as the values.
        """
        ...

Verify that the repository connects to the correct collection and that
`get_nationality_value_counts` returns the expected columns.

**Code 7.4.3.2**:

In [ ]:
from pymongo.collection import Collection

repo = MongoRepository() # instantiate MongoRepository
assert isinstance(repo.collection, Collection)
print("Collection:", repo.collection.name)

df_nat = repo.get_nationality_value_counts(normalize=False)
print("Columns:", sorted(df_nat.columns.tolist()))
df_nat.head()

✅ The collection name prints as `ds-applicants` and the nationality
DataFrame carries the four expected columns. The repository is wired to
the right data source — every higher layer can now trust it.

Test `get_ages` and `get_ed_value_counts`. You should see a Series of ages
and a Series of education counts ordered from lowest to highest degree.

**Code 7.4.3.3**:

In [ ]:
repo = MongoRepository()

ages = repo.get_ages()
print("Ages type:", type(ages).__name__, "| length:", len(ages))

degrees = repo.get_ed_value_counts(normalize=False)
print("Education index:", degrees.index.tolist())
degrees

📊 The education index comes back in *deliberate* order — High School →
Doctorate — not alphabetical or count-sorted. That ordering is baked into
the repository method so every chart built from it reads left-to-right as
increasing attainment.

Test `get_no_quiz_per_day`. The result should be a 30-element Series.

**Code Task 7.4.3.4**:

In [ ]:
repo = ...
no_quiz = repo. ...
print("Shape:", ...)
no_quiz.head()

### Checkpoint

In [ ]:
from pymongo.collection import Collection

repo = MongoRepository()

assert isinstance(repo.collection, Collection), (
    f"Expected Collection, got {type(repo.collection).__name__}"
)
assert repo.collection.name == "ds-applicants", (
    f"Expected collection 'ds-applicants', "
    f"got '{repo.collection.name}'"
)

df_nat = repo.get_nationality_value_counts(normalize=False)
assert isinstance(df_nat, pd.DataFrame), (
    f"get_nationality_value_counts should return DataFrame, "
    f"got {type(df_nat).__name__}"
)
nat_cols = sorted(df_nat.columns.tolist())
expected_cols = ["count", "country_iso2", "country_iso3", "country_name"]
assert nat_cols == expected_cols, (
    f"Expected columns {expected_cols}, got {nat_cols}"
)

ages = repo.get_ages()
assert isinstance(ages, pd.Series), (
    f"get_ages should return Series, got {type(ages).__name__}"
)

degrees = repo.get_ed_value_counts(normalize=False)
assert isinstance(degrees, pd.Series), (
    f"get_ed_value_counts should return Series, "
    f"got {type(degrees).__name__}"
)
expected_ed = [
    "High School or Baccalaureate",
    "Some College (1-3 years)",
    "Bachelor's degree",
    "Master's degree",
    "Doctorate (e.g. PhD)",
]
assert degrees.index.tolist() == expected_ed, (
    f"Education index mismatch.\n"
    f"Expected: {expected_ed}\n"
    f"Got: {degrees.index.tolist()}"
)

no_quiz = repo.get_no_quiz_per_day()
assert isinstance(no_quiz, pd.Series), (
    f"get_no_quiz_per_day should return Series, "
    f"got {type(no_quiz).__name__}"
)
assert no_quiz.shape == (30,), (
    f"Expected shape (30,), got {no_quiz.shape}"
)

print("All MongoRepository checks passed.")

## 4. Business Layer — Demographic Visualizations

### Problem

The dashboard displays three demographic charts based on a user's
drop-down selection. You need a class that takes data from the repository
and returns ready-to-display Plotly figures, keeping chart-building logic
separate from both data access and the Dash UI.

In a typical project this class would live in a `business.py` module
alongside other business-logic components, making it easy to reuse the
same charts in different dashboards or reports. Here you will define it
directly in the notebook.

🧱 **This is the middle tier.** `GraphBuilder` asks the repository for
data and turns it into figures. It never queries MongoDB itself and it
never talks to Dash — it sits cleanly between them.

### Approach

Define a `GraphBuilder` class with:

-   `__init__` — create and store a `MongoRepository` instance.
-   `build_nat_choropleth()` — fetch nationality data and return a Plotly
    choropleth map using `px.choropleth`.
-   `build_age_hist()` — fetch ages and return a histogram using
    `px.histogram`.
-   `build_ed_bar()` — fetch education counts and return a bar chart using
    `px.bar`.
-   `build_contingency_bar()` — fetch the contingency table and return a
    grouped bar chart. You will test this method later, after the
    experiment runner is in place.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284988", h="3298dbabb7", width=700, height=450) 

### Tasks

Define the `GraphBuilder` class. Each chart method should call the
appropriate repository method, build a figure, and return it.

🔍 Note the recurring shape of every method: **ask the repo for data →
build a figure → return it.** That uniformity is what lets a single Dash
callback swap one chart for another just by calling a different method.

**Code Task 7.4.4.1**:

In [ ]:
class GraphBuilder:
    """Builds Plotly figures using data from MongoRepository.

    This class provides chart-building methods for the three
    demographic visualizations and the contingency-table bar
    chart used in the Results section of the dashboard.

    Attributes
    ----------
    repo : MongoRepository
        Repository instance used to fetch data for each chart.
    """

    def __init__(self):
        """Instantiate a ``MongoRepository`` and store it.

        The repository is created once and reused by every
        chart-building method in this class.
        """
        self.repo = ...  # instantiate MongoRepository

    def build_nat_choropleth(self):
        """Build a choropleth map of applicant nationalities.

        Fetches normalized nationality counts from the
        repository and renders them on a world map using
        ``plotly.express.choropleth``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly choropleth figure colored by applicant
            proportion per country.
        """
        ...

    def build_age_hist(self):
        """Build a histogram of applicant ages.

        Fetches the age Series from the repository and
        renders a 30-bin histogram using
        ``plotly.express.histogram``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly histogram figure showing the age
            distribution.
        """
        ...

    def build_ed_bar(self):
        """Build a bar chart of education levels.

        Fetches raw education counts from the repository and
        renders a vertical bar chart using
        ``plotly.express.bar``.

        Returns
        -------
        plotly.graph_objects.Figure
            A Plotly bar chart figure with education levels on
            the x-axis and counts on the y-axis.
        """
        ...

    def build_contingency_bar(self):
        """Build a grouped bar chart from the contingency table.

        Fetches the 2x2 contingency table from the repository
        and renders a side-by-side (grouped) bar chart using
        ``plotly.express.bar``. This method should only be
        called after experiment data has been generated.

        Returns
        -------
        plotly.graph_objects.Figure
            A grouped bar chart showing quiz completion counts
            for each experiment group.
        """
        ...

Test each chart method. You should see the choropleth, histogram, and bar
chart rendered below.

**Code 7.4.4.2**:

In [ ]:
from plotly.graph_objects import Figure

gb = GraphBuilder() # instantiate GraphBuilder
fig_choro = gb.build_nat_choropleth() # build the choropleth
assert isinstance(fig_choro, Figure)
fig_choro.show()

📊 The choropleth shades each country by its share of applicants —
darker = more applicants. It's the same nationality data from Lesson 1,
now rendered as an interactive map the dashboard can display on demand.

**Code 7.4.4.3**:

In [ ]:
fig_age = gb.build_age_hist() # build the age histogram
assert isinstance(fig_age, Figure)
fig_age.show()

**Code 7.4.4.4**:

In [ ]:
fig_ed = gb.build_ed_bar() # build the education bar chart
assert isinstance(fig_ed, Figure)
fig_ed.show()

✅ All three demographic charts render from a single `GraphBuilder`
instance. Each pulls its own data through the repository, confirming the
two layers cooperate exactly as the architecture intends.

### Checkpoint

In [ ]:
from plotly.graph_objects import Figure

gb = GraphBuilder()

for name in ["build_nat_choropleth", "build_age_hist", "build_ed_bar"]:
    fig = getattr(gb, name)()
    assert isinstance(fig, Figure), (
        f"{name} should return Figure, got {type(fig).__name__}"
    )

print("All GraphBuilder checks passed.")

## 5. Business Layer — Statistical Computations

### Problem

The Experiment and Results sections of the dashboard require statistical
computations that do not belong in the presentation layer: calculating
required sample sizes, estimating observation probabilities, running
experiments, and performing chi-square tests. A dedicated `StatsBuilder`
class keeps this logic organized.

Like `GraphBuilder`, this class would normally live in the `business.py`
module so it can be imported by any notebook or application that needs
these computations. You will define it in a notebook cell for convenience.

➡️ This class is where the **statistics from Lesson 3** get packaged for
reuse: power analysis (`calculate_n_obs`), the CLT/CDF duration estimate
(`calculate_cdf_pct`), the experiment runner, and the chi-square test
(`run_chi_square`). The math is identical — recall the per-group sample
size solves the power equation at $\alpha = 0.05$, power $= 0.8$, and the
CDF uses the CLT total $N(d\mu,\, \sigma\sqrt{d})$ — but now each formula
hides behind a clean method name.

### Approach

Define a `StatsBuilder` class with:

-   `__init__` — create and store a `MongoRepository` instance.
-   `calculate_n_obs(effect_size)` — use your power-analysis code from
    Lesson 7.3 to compute the required observations per group. Return an
    `int`.
-   `calculate_cdf_pct(n_obs, days)` — use data from the repository's
    `get_no_quiz_per_day` and your CDF logic from Lesson 7.3 to return a
    `float` (percentage).
-   `run_experiment(days)` — use the `Experiment` class from
    `wqulibs.ab_test.experiment` to simulate running the experiment for
    the given number of days.
-   `run_chi_square()` — fetch the contingency table from the repository
    and run a chi-square test using `statsmodels`. Return the result
    object (a `_Bunch`).

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1183284321", h="3298dbabb7", width=700, height=450) 

### Tasks

Define the `StatsBuilder` class.

**Code Task 7.4.5.1**:

In [ ]:
from wqulibs.ab_test.experiment import Experiment
from scipy import stats as st
from statsmodels.stats.contingency_tables import Table

class StatsBuilder:
    """Computes statistics for experiment monitoring.

    Provides methods for power analysis, experiment-duration
    estimation, experiment execution, and hypothesis testing.
    All data access is delegated to a ``MongoRepository``
    instance created on construction.

    Attributes
    ----------
    repo : MongoRepository
        Repository instance used to fetch experiment data.
    """

    def __init__(self):
        """Instantiate a ``MongoRepository`` and store it.

        The repository is created once and shared across all
        statistical methods in this class.
        """
        self.repo = ...  # instantiate MongoRepository

    def calculate_n_obs(self, effect_size):
        """Calculate the required number of observations per group.

        Uses the ``GofChisquarePower`` class from statsmodels
        to solve for the minimum sample size needed to detect
        the given ``effect_size`` with alpha = 0.05 and
        power = 0.8 in a two-bin chi-square test. The raw
        per-bin result is rounded up with ``math.ceil`` and
        then doubled to give the total observations needed
        across both groups.

        Parameters
        ----------
        effect_size : float
            The minimum detectable effect size (Cohen's *w*).
            Common benchmarks: 0.2 (small), 0.5 (medium),
            0.8 (large).

        Returns
        -------
        int
            Total number of observations required (both groups
            combined).
        """
        ...

    def calculate_cdf_pct(self, n_obs, days):
        """Estimate the probability of collecting enough data.

        Retrieves daily no-quiz applicant counts from the
        repository, computes the mean and standard deviation,
        and uses the central limit theorem to model the total
        arrivals over the specified number of ``days``. Returns
        the probability (as a percentage) that the total will
        reach or exceed ``n_obs``.

        Parameters
        ----------
        n_obs : int
            Target number of observations to collect.
        days : int
            Number of days the experiment would run.

        Returns
        -------
        float
            Probability (0–100) of reaching ``n_obs``
            observations within the given ``days``.
        """
        ...

    def run_experiment(self, days):
        """Run a simulated A/B experiment.

        Creates an ``Experiment`` instance connected to
        ``self.repo`` and runs it for the specified number
        of days. The generated student records are inserted
        into the MongoDB collection.

        Parameters
        ----------
        days : int
            Number of simulated days to run the experiment.
        """
        ...

    def run_chi_square(self):
        """Perform a chi-square test on the experiment results.

        Fetches the 2x2 contingency table from the repository,
        wraps it in a ``statsmodels.stats.contingency_tables.Table``,
        and runs ``test_nominal_association`` to test whether
        the experiment group and quiz completion are independent.

        Returns
        -------
        statsmodels.stats.contingency_tables._Bunch
            Test result containing the chi-square statistic,
            p-value, and degrees of freedom.
        """
        ...

Test `calculate_n_obs`. For an effect size of 0.2, the result should be
394.

🧮 That 394 is the power equation solved for $n$ at effect size $w = 0.2$,
$\alpha = 0.05$, power $= 0.8$, then doubled for the two groups — exactly
the calculation you did by hand in Lesson 3, now returned by one method
call.

**Code 7.4.5.2**:

In [ ]:
sb = StatsBuilder() # instantiate StatsBuilder
n_obs = sb.calculate_n_obs(effect_size=0.2)
print("Required observations per group:", n_obs)

Test `calculate_cdf_pct`. For 394 observations over 12 days the
probability should exceed 95%.

**Code 7.4.5.3**:

In [ ]:
pct = sb.calculate_cdf_pct(n_obs=394, days=12)
print(f"Probability of enough observations: {pct}%")

🔍 The percentage answers the planning question: *"If I run for 12 days,
how likely am I to collect my 394 observations?"* Above 95% means the
duration is comfortably sufficient — the CLT total's mean ($d\mu$)
clears the target with room to spare.

Test `run_experiment`. After running for 1 day the collection should
contain more documents than before. The cell also resets the experiment
afterward so later sections start clean.

**Code 7.4.5.4**:

In [ ]:
mr = MongoRepository()
exp = Experiment(repo=mr)
exp.reset_experiment()

docs_before = mr.collection.count_documents({})
sb.run_experiment(days=1)
docs_after = mr.collection.count_documents({})
print("Documents added:", docs_after - docs_before)

exp.reset_experiment()

Test `run_chi_square`. Run a short experiment first so there is
contingency data, then verify the result type.

**Code 7.4.5.5**:

In [ ]:
sb.run_experiment(days=1)
result = sb.run_chi_square() # run chi-square test
print("p-value:", result.pvalue)

🔍 With only 1 simulated day the sample is tiny, so don't read much into
this particular p-value — the point is that the method returns a proper
test result the dashboard can surface. A real run uses the duration from
the slider.

Now also verify the contingency bar chart from `GraphBuilder`, since
experiment data now exists.

**Code 7.4.5.6**:

In [ ]:
gb = GraphBuilder()
fig_ct = gb.build_contingency_bar() # build the contingency bar chart
assert isinstance(fig_ct, Figure)
fig_ct.show()

✅ `build_contingency_bar` works now that experiment documents exist —
the same method that sat untested in section 4. Both business-layer
classes are complete and verified.

### Checkpoint

In [ ]:
from wqulibs.ab_test.experiment import Experiment
from statsmodels.stats.contingency_tables import _Bunch

sb = StatsBuilder()

# calculate_n_obs
n_obs = sb.calculate_n_obs(effect_size=0.2)
assert isinstance(n_obs, int), (
    f"calculate_n_obs should return int, got {type(n_obs).__name__}"
)
assert n_obs == 394, (
    f"Expected 394 for effect_size=0.2, got {n_obs}"
)

# calculate_cdf_pct
pct = sb.calculate_cdf_pct(n_obs=394, days=12)
assert isinstance(pct, float), (
    f"calculate_cdf_pct should return float, "
    f"got {type(pct).__name__}"
)
assert 95 < pct <= 100, (
    f"Expected pct in (95, 100], got {pct}"
)

# run_experiment
mr = MongoRepository()
exp = Experiment(repo=mr)
exp.reset_experiment()
docs_before = mr.collection.count_documents({})
sb.run_experiment(days=1)
docs_after = mr.collection.count_documents({})
assert docs_after > docs_before, (
    "run_experiment should add documents to the collection"
)

# run_chi_square
result = sb.run_chi_square()
assert isinstance(result, _Bunch), (
    f"run_chi_square should return _Bunch, "
    f"got {type(result).__name__}"
)

exp.reset_experiment()
print("All StatsBuilder checks passed.")

## 6. Dashboard Layout and Callbacks

### Problem

With the database and business layers ready, you can now build the
presentation layer: the Dash application layout and the callback functions
that connect user interactions to the classes you just created. This is
where everything comes together into a single interactive experience.

🧱 **This is the top tier.** The layout declares *what the user sees*; the
callbacks declare *what happens when they interact*. Crucially, the
callbacks contain almost no logic of their own — they just call
`GraphBuilder` and `StatsBuilder` and route the results into the page.
That thinness is the payoff of the architecture.

### Approach

In a single cell, you will:

1.  Create a `Dash` application instance called `app`.

2.  Define `app.layout` with three `H1` sections and the following
    components:

    **Applicant Demographics:**

    -   `dcc.Dropdown` with id `"demo-plots-dropdown"` and options
        `"Nationality"`, `"Age"`, `"Education"`.
    -   `html.Div` with id `"demo-plots-display"`.

    **Experiment:**

    -   `dcc.Slider` with id `"effect-size-slider"` (range: 0.1–1.0,
        step: 0.1, default: 0.2).
    -   `html.Div` with id `"effect-size-display"`.
    -   `dcc.Slider` with id `"experiment-days-slider"` (range: 1–30,
        step: 1, default: 10).
    -   `html.Div` with id `"experiment-days-display"`.

    **Results:**

    -   `html.Button` with id `"start-experiment-button"` and label
        `"Run Experiment"`.
    -   `html.Div` with id `"results-display"`.

3.  Define four callbacks:

    -   `display_demo_graph` — reads `"demo-plots-dropdown"`, builds the
        appropriate chart via `GraphBuilder`, and outputs a `dcc.Graph`
        to `"demo-plots-display"`.
    -   `display_group_size` — reads `"effect-size-slider"`, computes
        sample size via `StatsBuilder`, and outputs text to
        `"effect-size-display"`.
    -   `display_cdf_pct` — reads `"experiment-days-slider"` and
        `"effect-size-slider"`, computes the CDF percentage, and outputs
        text to `"experiment-days-display"`.
    -   `display_results` — reads `"start-experiment-button"` (as `Input`)
        and `"experiment-days-slider"` (as `State`), runs the experiment,
        builds the contingency bar chart, runs the chi-square test, and
        outputs the chart and test summary to `"results-display"`.

> 📌 **Tip:** The Results callback uses `State` for the days slider (not
> `Input`) because you only want the experiment to run when the **button**
> is clicked — not every time the slider nudges. `State` reads a value
> *without* triggering the callback; `Input` both reads *and* triggers.

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1105512918", h="3298dbabb7", width=700, height=450)

### Tasks

Build the complete Dash application. This is the largest single cell in
the notebook. Read through the scaffold carefully — each section is marked
with comments.

**Code Task 7.4.6.1**:

In [ ]:
from dash import html

# Create the Dash application
app = ...  # instantiate Dash(__name__)

# Define the layout
app.layout = html.Div([
    # --- Applicant Demographics ---
    html.H1("Applicant Demographics"),
    # Add Dropdown (id="demo-plots-dropdown") and
    # Div (id="demo-plots-display") here
    ...,

    # --- Experiment ---
    html.H1("Experiment"),
    # Add effect-size Slider (id="effect-size-slider"),
    # Div (id="effect-size-display"),
    # days Slider (id="experiment-days-slider"),
    # Div (id="experiment-days-display") here
    ...,

    # --- Results ---
    html.H1("Results"),
    # Add Button (id="start-experiment-button") and
    # Div (id="results-display") here
    ...,
])

# Callback: demographic chart
@app.callback(
    Output("demo-plots-display", "children"),
    Input("demo-plots-dropdown", "value"),
)
def display_demo_graph(selection):
    """Return the selected demographic chart.

    Uses a ``GraphBuilder`` to construct the chart that
    matches the user's drop-down selection.

    Parameters
    ----------
    selection : str
        One of ``"Nationality"``, ``"Age"``, or
        ``"Education"``.

    Returns
    -------
    dash.dcc.Graph
        A Dash ``Graph`` component wrapping the Plotly figure.
    """
    ...

# Callback: group size from effect-size slider
@app.callback(
    Output("effect-size-display", "children"),
    Input("effect-size-slider", "value"),
)
def display_group_size(effect_size):
    """Display the required sample size for the given effect size.

    Instantiates a ``StatsBuilder``, calls ``calculate_n_obs``,
    and returns a paragraph showing the result.

    Parameters
    ----------
    effect_size : float
        Value from the effect-size slider (0.1–1.0).

    Returns
    -------
    dash.html.P
        A paragraph element stating the required observations.
    """
    ...

# Callback: CDF percentage from both sliders
@app.callback(
    Output("experiment-days-display", "children"),
    Input("experiment-days-slider", "value"),
    Input("effect-size-slider", "value"),
)
def display_cdf_pct(days, effect_size):
    """Display the probability of collecting enough observations.

    Instantiates a ``StatsBuilder``, computes the required
    sample size and the CDF percentage for the given number
    of days, and returns a paragraph with the result.

    Parameters
    ----------
    days : int
        Value from the experiment-days slider (1–30).
    effect_size : float
        Value from the effect-size slider (0.1–1.0).

    Returns
    -------
    dash.html.P
        A paragraph element stating the probability percentage.
    """
    ...

# Callback: run experiment and show results
@app.callback(
    Output("results-display", "children"),
    Input("start-experiment-button", "n_clicks"),
    State("experiment-days-slider", "value"),
)
def display_results(n_clicks, days):
    """Run the experiment and display results.

    Triggered when the user clicks the "Run Experiment"
    button. Runs the A/B experiment for the specified
    number of days, builds the contingency-table bar chart,
    performs the chi-square test, and returns a ``Div``
    containing the chart and test summary.

    Parameters
    ----------
    n_clicks : int or None
        Number of times the button has been clicked.
        ``None`` on initial page load.
    days : int
        Value from the experiment-days slider (via ``State``).

    Returns
    -------
    dash.html.Div or dash.html.P
        A ``Div`` containing the bar chart and test results,
        or a placeholder ``P`` if the button has not been
        clicked yet.
    """
    ...

🧠 **Trace one interaction end-to-end.** A user drags the effect-size
slider → its `value` changes → Dash fires `display_group_size` → the
callback builds a `StatsBuilder`, calls `calculate_n_obs`, and returns an
`html.P` → Dash drops that paragraph into `"effect-size-display"`. Three
layers, one slider drag — and the callback itself is barely four lines.
That is the architecture doing its job.

### Checkpoint

In [ ]:
# Verify app exists and has key components
assert hasattr(app, "layout"), "app must have a layout attribute"

# Flatten layout to string representation for ID checks
layout_str = str(app.layout)

expected_ids = [
    "demo-plots-dropdown",
    "demo-plots-display",
    "effect-size-slider",
    "effect-size-display",
    "experiment-days-slider",
    "experiment-days-display",
    "start-experiment-button",
    "results-display",
]

for comp_id in expected_ids:
    assert comp_id in layout_str, (
        f"Component with id '{comp_id}' not found in layout"
    )

print("All layout checks passed.")

## 7. Launch the Dashboard

### Problem

With all layers defined, the final step is to start the Dash server so you
can interact with the complete dashboard. The app will render inline,
directly below the launch cell.

### Approach

Call
`app.run(host="0.0.0.0", port=9000, jupyter_mode="inline", debug=True)`
to start the server. The dashboard will appear as an embedded iframe in
the notebook. Interact with the drop-down, sliders, and button to verify
that all callbacks work correctly.

> 📌 **Reminder:** If something looks wrong, restart the kernel and rerun
> all cells from the top. Only one Dash server can run per kernel session.

### Tasks

Launch the dashboard. After the cell runs, you should see the full
application rendered below with all three sections.

**Code 7.4.7.1**:

In [ ]:
app.run(host="0.0.0.0", port=9000, jupyter_mode="inline", debug=True)

### Checkpoint

This is a visual checkpoint. Verify the following by interacting with the
dashboard:

-   Selecting "Nationality", "Age", or "Education" in the drop-down
    renders the corresponding chart.
-   Moving the effect-size slider updates the required number of
    observations.
-   Moving the days slider (and effect-size slider) updates the
    probability percentage.
-   Clicking "Run Experiment" displays a contingency bar chart and
    chi-square test results.

⚠️ **A closing word on judgment.** The dashboard makes it effortless to
re-run the experiment and read a p-value — which is exactly why a careful
analyst stays disciplined. Resist clicking "Run Experiment" repeatedly and
stopping at the first significant result (that's the *peeking* fallacy
from Lesson 3, which inflates false positives). And remember that a
significant p-value tells you the email *had an effect*, not that the
effect is *large enough to be worth the cost* — read the contingency bars
and odds ratio alongside the p-value before recommending a rollout.

In [ ]:
print(
    "Visual checkpoint: interact with the dashboard above\n"
    "and verify all four behaviors described in the list."
)

# Wrap-up

In this lesson you:

-   Learned how the three-tier architecture (database, business logic,
    presentation) organizes interactive applications, keeping data access,
    computation, and UI concerns separate.
-   Built a `MongoRepository` class that encapsulates MongoDB queries for
    applicant demographics, daily quiz statistics, and contingency tables.
-   Created a `GraphBuilder` class that produces interactive Plotly
    choropleth maps, histograms, bar charts, and contingency-table
    visualizations.
-   Implemented a `StatsBuilder` class that computes required sample sizes,
    CDF probabilities, runs A/B experiments, and performs chi-square tests.
-   Assembled a complete Dash dashboard with drop-downs, sliders, buttons,
    and callbacks that connect user interactions to the business-layer
    classes.
-   Launched the dashboard inline and verified that all components work
    end-to-end.

This concludes Project 7. Next, you will begin **Project 8**, where you
will integrate the full range of skills developed throughout the Lab —
combining data engineering, machine learning, experimentation, and
communication into a final capstone-style project.